In [ ]:
# CELL 1: Cài thư viện, cấu hình và thử model nhận diện XE.
# Chạy theo thứ tự 1 -> 2 -> 3. Sửa ANH thành đường dẫn ảnh của bạn.
# Nếu thiếu thư viện, bỏ dấu # ở dòng sau và chạy lại cell:
# %pip install -U ultralytics matplotlib pillow
# Nguồn model xe: https://huggingface.co/Ultralytics/YOLOv8
# Hai model chỉ phát hiện khung bao, chưa OCR hoặc xác định vượt đèn đỏ.
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from ultralytics import YOLO

ANH = Path(r"test.jpg")  # Ví dụ: Path(r"C:\Users\Khiem\Pictures\xe.jpg")
MODEL_XE = "yolov8n.pt"  # Tự tải model chính thức khi chưa có; cần Internet lần đầu.
MODEL_BIEN_SO = Path("DetectLisence_YOLO11.pt")
CONF_XE, CONF_BIEN_SO = 0.25, 0.25
THIET_BI = 0 if torch.cuda.is_available() else "cpu"
THU_MUC_KET_QUA = Path("ket_qua")
THU_MUC_KET_QUA.mkdir(exist_ok=True)

if not ANH.is_file():
    raise FileNotFoundError(f"Không thấy ảnh: {ANH.resolve()}. Hãy sửa biến ANH.")
with Image.open(ANH) as im:
    anh_rgb = np.array(ImageOps.exif_transpose(im).convert("RGB"))
anh_bgr = cv2.cvtColor(anh_rgb, cv2.COLOR_RGB2BGR)

def hien_thi_luu(anh, tieu_de, ten_file):
    rgb = cv2.cvtColor(anh, cv2.COLOR_BGR2RGB)
    Image.fromarray(rgb).save(THU_MUC_KET_QUA / ten_file)
    plt.figure(figsize=(14, 9))
    plt.imshow(rgb)
    plt.title(tieu_de)
    plt.axis("off")
    plt.show()
    print("Đã lưu:", (THU_MUC_KET_QUA / ten_file).resolve())

model_xe = YOLO(MODEL_XE)
ket_qua_xe = model_xe.predict(
    anh_bgr, classes=[2, 3, 5, 7], conf=CONF_XE,
    imgsz=640, device=THIET_BI, verbose=False
)[0]  # COCO: car=2, motorcycle=3, bus=5, truck=7
print(f"Thiết bị: {THIET_BI} | Phát hiện {len(ket_qua_xe.boxes)} xe")
hien_thi_luu(ket_qua_xe.plot(), "Model 1: Phat hien xe", "01_xe.jpg")


In [ ]:
# CELL 2: Thử model BIỂN SỐ trên toàn ảnh. Chạy cell 1 trước.
# Nguồn: https://github.com/ddkhoa2605/LicensePlateDetection
# Model phát hiện vị trí biển số, KHÔNG đọc ký tự.
# Nếu chưa có file, tải model từ repository tác giả (cần Internet).
from urllib.request import urlopen
import shutil

if not MODEL_BIEN_SO.is_file():
    url = ("https://raw.githubusercontent.com/ddkhoa2605/"
           "LicensePlateDetection/main/DetectLisence_YOLO11.pt")
    MODEL_BIEN_SO.parent.mkdir(parents=True, exist_ok=True)
    tam = MODEL_BIEN_SO.with_suffix(".download")
    try:
        print("Đang tải model biển số...")
        with urlopen(url, timeout=90) as response, tam.open("wb") as f:
            shutil.copyfileobj(response, f)
        if tam.stat().st_size < 100_000:
            raise ValueError("File tải về quá nhỏ, có thể là Git LFS pointer hoặc trang lỗi.")
        tam.replace(MODEL_BIEN_SO)
    except Exception as e:
        tam.unlink(missing_ok=True)
        raise RuntimeError(
            "Không tải được model. Tải DetectLisence_YOLO11.pt từ GitHub tác giả, "
            "đặt cạnh notebook hoặc sửa MODEL_BIEN_SO ở cell 1."
        ) from e

model_bien_so = YOLO(str(MODEL_BIEN_SO))
ket_qua_bien_so = model_bien_so.predict(
    anh_bgr, conf=CONF_BIEN_SO, imgsz=640,
    device=THIET_BI, verbose=False
)[0]
print(f"Phát hiện {len(ket_qua_bien_so.boxes)} biển số")
hien_thi_luu(ket_qua_bien_so.plot(), "Model 2: Phat hien bien so", "02_bien_so.jpg")


In [ ]:
# CELL 3: Kết hợp hai model: tìm xe -> cắt từng xe -> tìm biển số trong ảnh xe.
# Chạy cell 1 và 2 trước. Đây là kết hợp pipeline, KHÔNG ghép trọng số .pt.
# Nhãn xe #1, #2... chỉ là số thứ tự trong ảnh, không phải ID tracking.
from IPython.display import display

ket_qua_xe = model_xe.predict(
    anh_bgr, classes=[2, 3, 5, 7], conf=CONF_XE,
    imgsz=640, device=THIET_BI, verbose=False
)[0]
anh_ket_hop = anh_bgr.copy()
H, W = anh_bgr.shape[:2]
danh_sach_bien_so = []

def ve_khung(anh, toa_do, nhan, mau):
    x1, y1, x2, y2 = map(int, toa_do)
    cv2.rectangle(anh, (x1, y1), (x2, y2), mau, 2)
    cv2.putText(anh, nhan, (x1, max(18, y1 - 7)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, mau, 2)

for stt, box in enumerate(ket_qua_xe.boxes, start=1):
    xy = box.xyxy[0].cpu().numpy()
    x1, y1 = np.floor(xy[:2]).astype(int)
    x2, y2 = np.ceil(xy[2:]).astype(int)
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
    if x2 <= x1 or y2 <= y1:
        continue
    loai_xe = model_xe.names[int(box.cls.item())]
    ve_khung(anh_ket_hop, (x1, y1, x2, y2),
             f"{loai_xe} #{stt} {box.conf.item():.2f}", (0, 200, 0))
    anh_xe = anh_bgr[y1:y2, x1:x2].copy()
    bien_so_trong_xe = model_bien_so.predict(
        anh_xe, conf=CONF_BIEN_SO, imgsz=640,
        device=THIET_BI, verbose=False
    )[0]
    for so, plate in enumerate(bien_so_trong_xe.boxes, start=1):
        p = plate.xyxy[0].cpu().numpy()
        px1, py1 = np.floor(p[:2]).astype(int)
        px2, py2 = np.ceil(p[2:]).astype(int)
        # Chuyển tọa độ ảnh xe về tọa độ ảnh gốc.
        bx1, by1 = max(x1, x1 + px1), max(y1, y1 + py1)
        bx2, by2 = min(x2, x1 + px2), min(y2, y1 + py2)
        if bx2 <= bx1 or by2 <= by1:
            continue
        ve_khung(anh_ket_hop, (bx1, by1, bx2, by2),
                 f"plate #{stt}.{so} {plate.conf.item():.2f}", (0, 0, 255))
        crop = anh_bgr[by1:by2, bx1:bx2].copy()
        duong_dan = THU_MUC_KET_QUA / f"bien_so_xe_{stt}_{so}.jpg"
        Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).save(duong_dan)
        danh_sach_bien_so.append((stt, crop, duong_dan))

print(f"Phát hiện {len(ket_qua_xe.boxes)} xe, {len(danh_sach_bien_so)} biển số trong xe.")
hien_thi_luu(anh_ket_hop, "Ket hop: xe (xanh) va bien so (do)", "03_ket_hop.jpg")
if not danh_sach_bien_so:
    print("Chưa tìm được biển số trong xe. Kiểm tra cell 1/2; dùng ảnh rõ hơn hoặc điều chỉnh conf.")
else:
    for stt, crop, duong_dan in danh_sach_bien_so:
        print(f"Ảnh biển số của xe #{stt}: {duong_dan}")
        display(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
